# Kalman filters — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Blend prediction and measurement using uncertainty as the weight
Kalman filters update a Gaussian state through linear dynamics and Gaussian observations. These examples compute prediction, gain, correction, repeated filtering, and the effect of measurement noise.

In [ ]:
# 1) predict step adds process noise
m,P=0.,1.; A,Q=1.,0.2; mp=A*m; Pp=A*P*A+Q
plt.figure(figsize=(6,3)); plt.bar(['prior var','predicted var'],[P,Pp]); plt.title('process noise widens uncertainty')
assert abs(Pp-1.2)<1e-12

In [ ]:
# 2) update step uses Kalman gain
z,R=1.4,0.5; K=Pp/(Pp+R); mu=mp+K*(z-mp); Pu=(1-K)*Pp
plt.figure(figsize=(6,3)); plt.bar(['prediction','measurement','posterior'],[mp,z,mu]); plt.title(f'gain={K:.3f}')
assert abs(K-0.7058823529411765)<1e-12 and abs(mu-0.988235294117647)<1e-12

In [ ]:
# 3) posterior variance shrinks after seeing z
plt.figure(figsize=(6,3)); plt.bar(['predicted','updated'],[Pp,Pu]); plt.title('measurement narrows uncertainty')
assert Pu<Pp and abs(Pu-0.35294117647058826)<1e-12

In [ ]:
# 4) repeated filtering tracks noisy observations
obs=[1.4,1.8,2.0,2.2]; m,P=0.,1.; ms=[]
for z in obs:
    P=P+Q; K=P/(P+R); m=m+K*(z-m); P=(1-K)*P; ms.append(m)
plt.figure(figsize=(6,3)); plt.plot(obs,'o--',label='obs'); plt.plot(ms,'-o',label='filter'); plt.legend(); plt.title('recursive Gaussian belief')
assert ms[-1]>ms[0]

In [ ]:
# 5) noisier sensor lowers the Kalman gain
Rs=[0.1,0.5,2.0]; gains=[Pp/(Pp+r) for r in Rs]
plt.figure(figsize=(6,3)); plt.plot(Rs,gains,'-o'); plt.xlabel('measurement noise R'); plt.ylabel('gain'); plt.title('trust noisy sensors less')
assert gains[0]>gains[-1]